# 01 Data Audit and Manifest Preparation

This notebook audits the downloaded public datasets and identifies whether each dataset contains images, masks, annotations, metadata, or only classification-style folders.

The goal is to determine which datasets are actually usable for optic disc/cup segmentation.

In [ ]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

PROJECT_ROOT

In [ ]:
DATASET_KEYS = [
    "glaucoma_fundus_imaging_bundle",
    "papila",
]

DATASET_KEYS

In [ ]:
from glaucoma_segmentation.data.discover_files import inventory_registry_datasets

inventory_df, summary_df, candidate_df = inventory_registry_datasets(
    dataset_keys=DATASET_KEYS,
    save_reports=True,
)

summary_df

In [ ]:
candidate_df[
    [
        "dataset_key",
        "relative_path",
        "extension",
        "file_class",
        "has_mask_keyword",
        "file_size_bytes",
    ]
].head(100)

In [ ]:
summary_df.sort_values(
    ["dataset_key", "file_count"],
    ascending=[True, False],
)

In [ ]:
dataset_summary = (
    inventory_df
    .groupby(["dataset_key", "file_class"])
    .size()
    .reset_index(name="file_count")
    .pivot(index="dataset_key", columns="file_class", values="file_count")
    .fillna(0)
    .astype(int)
)

dataset_summary

## Initial audit complete

The next step is to review the candidate masks/annotations and determine which datasets can support optic disc/cup segmentation.

After that, the manifest-building portion of this notebook will create structured image/mask pair records.